# The review's convex combination beside our bank

At the meeting where `kf-bank-q-sweep.ipynb` and `awu-gisv-vs-banks.ipynb` were shown, Augusto said that the bank of
filters already exists, and Leszek asked for references and a comparison. Augusto sent J. Arenas-García,
L. A. Azpicueta-Ruiz, M. T. M. Silva, V. H. Nascimento and A. H. Sayed, "Combinations of adaptive filters: performance and
convergence properties", *IEEE Signal Processing Magazine*, January 2016 (below: the review), and pointed to its convex
combination with "weight sharing", gradual transfer first, all of them if possible.

**The review against our bank.**

| | the review | our bank (Algorithms 1-3) |
|---|---|---|
| weights | one mixing parameter $\lambda_t$, learned by a stochastic gradient on the **squared** output error $e_t^2$, through a sigmoid (cvx-PN-LMS, its Table 4) | the posterior weights $\mu_{t,j}$, from each filter's predictive density, eq. (bank.weights) |
| sharing | rules set by hand (its Table 5): gradual transfer, simple copy, feedback | the mixing step, eq. (bank.mixing), at every step, through $q$; it mixes the variances too |
| filters | mostly two, a fast and a slow one | $K = 6$ |

Our bank is the interacting multiple model (IMM) estimator of Blom and Bar-Shalom (1988) for $K$ random-walk models
that differ in $\varepsilon$: with the Gaussian $\chi$, Algorithm 3 is a textbook IMM to $2 \times 10^{-15}$
(`kf-bank-q-sweep.ipynb`, check 4.5).

**The test.** The same two filters under both combination layers, so that only the layer differs: a fast and a slow sKF
with the exact Laplacian $\chi$, combined (a) by the review's convex combination with each sharing scheme of its Table 5,
and (b) by our bank, Algorithm 1 with $K = 2$. Beside them the two sKFs alone and our bank of six sKFs.

**Choices.** Ramiro chose this test (convex combination only, all four sharing schemes, our main scenario) on
2026-10-15, then asked for a run with Gaussian noise beside it, where the squared error is the right cost. The details
below were set by Claude from the defaults proposed to him; he did not review them before the run.

| doubt | choice |
|---|---|
| the two sKFs | slow: the exact sKF at $-20$ dB of the earlier notebooks ($\varepsilon = 1.3493 \times 10^{-6}$ with $\beta^* = 0.2$; $10^{-7}$ with $\beta^* = 2$, the deepest settled point of the search of `kf-bank-q-sweep.ipynb`, $-18.7$ dB, read from its saved search). Fast: $\varepsilon = 10^{-4}$, the top of the sKF bank's grid. Both with the exact Laplacian $\chi$, $b_\eta = \mathrm{E}\lvert\eta_t\rvert$, $v_0 = 2$ |
| the combination | cvx-PN-LMS as in the review's Table 4: $a^+ = 4$, $\eta = 0.9$ (its Fig. 5), $a_0 = 0$. The affine combination is not run |
| sharing | none, and the three schemes of Table 5 as the review writes them (papers [47]-[49] themselves not read). Threshold $\lambda_0$ of gradual transfer and simple copy swept (below). Only $\boldsymbol w$ is shared; each sKF keeps its own $v$, which plays the role of the step size of the review's LMS filters |
| tuning | each scheme at every $\mu_a \in \{0.01, 0.03, 0.1, 0.3, 1, 3\}$ (the review uses 0.25 to 1) and every value of its parameters, $\ell \in \{0.5, 0.9, 0.99\}$, $N_0 \in \{2, 10, 100\}$, and for gradual transfer and simple copy $\lambda_0 \in \{0.98, 0.9, 0.7, 0.5\}$; the bank of the two sKFs at $q \in \{0, 10^{-4}, 10^{-3}, 10^{-2}, 0.1, 1/3\}$. Each is reported at its best setting, the lowest mean over the record (mean curves), as the banks' best $q$ before; every setting is printed. $\lambda_0 = 0.98$ (just below $\mathrm{sgm}(4) = 0.982$, the largest $\lambda_t$; the review's Fig. 8 uses 0.982) was the only value of the first run; there gradual transfer and simple copy fired almost only in the first few hundred steps, where $\lambda_t$ jumps between its bounds while the two filters are still nearly equal, and gave the numbers of "none" for $\mu_a \le 0.3$. 0.9 and 0.7 were added so that the schemes are not judged by a threshold they never reach. In that second run the best settings of the combination sat at $\mu_a = 1$ and $\lambda_0 = 0.7$, the edges of the grid, in both noises, so a third run added $\mu_a = 3$ and $\lambda_0 = 0.5$ |
| ours beside | the bank of the two sKFs (new runs); the slow sKF and the bank of six sKFs at $q = 10^{-4}$ and at its best $q$, read from `results/kf-bank-q-sweep/` |
| scenarios | notebook 09's change test with AR($-0.9$) input: $\beta^* = 0.2$ as before, and $\beta^* = 2$ (Gaussian). Under Gaussian noise ours keep the Laplacian likelihood, as in `kf-bank-q-sweep.ipynb`: there the bank's weights use the wrong density and the combination's cost is the right one |
| measures | those of `kf-bank-q-sweep.ipynb`: misalignment and excess error, from the mean and the median over the 20 realisations; and the weight of the fast filter ($\lambda_t$, or $\mu_t$ of the fast sKF in the bank) |
| where | this new notebook; its runs saved under `results/cvx-combination-vs-bank/` (git-ignored) as they finish |

**Not run.** The affine combination; more than two filters (the review's softmax and hierarchical schemes); KF
components; white input; the review's own figures.

**Run time.** The 350 runs of this notebook took 146 min when they were run (2026-10-15, in three passes as the grid
grew, see the tuning row above). Run again with `results/cvx-combination-vs-bank/` and `results/kf-bank-q-sweep/` in
place, the notebook takes about 10 s. Without `results/kf-bank-q-sweep/`, the copied code recomputes the runs of ours it
reads (not measured; a KF search of this kind, inside `tuned_values`, took about 13 min in `bank-kf-excess-error.ipynb`).

## 1. Imports

In [ ]:
import pickle
import time
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt
from scipy.ndimage import uniform_filter1d
from scipy.signal import lfilter
from scipy.special import gammaln, log_ndtr, logsumexp
from scipy.stats import gennorm
import rir_generator as rir

%config InlineBackend.figure_format = 'retina'
NOTEBOOK_START = time.time()
print("imports ready")

## 2. The scenario

Section 2 of `awu-gisv-vs-banks.ipynb` (commit `fab8cf0`), nothing edited: notebook 09's constants and signals, with the
AR coefficient and the noise shape `beta` as parameters.

In [ ]:
# === copied from awu-gisv-vs-banks.ipynb section 2 (commit fab8cf0), NOT edited ===
# === copied from kf-bank-q-sweep.ipynb section 2 (commit 56108a2), NOT edited ===
# === copied from bank-kf-excess-error.ipynb section 2 (commit f2ded06), which copied it from bank-skf-fkf.ipynb
#     (57bd568); one option added: the noise shape beta, default BETA (the same draws as before) ===
M = 128                     # filter length / length of the impulse response
N = 96000                   # steps per run at rest; the change run has 2N
R = 20                      # independent realisations
WARMUP = 500                # AR samples discarded so the input starts stationary
BETA = 0.2                  # shape of the generalized Gaussian measurement noise
SNR_DB = 5.0                # nominal signal-to-noise ratio
VAR_THETA_0 = 2.0           # initial prior variance on each weight (ours; Leszek's is 1/M)
FS = 8000                   # sampling rate [Hz]
TARGET_DB = -20.0           # floor the single filters are tuned to
BETA_GAUSS = 2.0            # the generalized Gaussian with beta = 2 is the Gaussian
DRIFT_DB = 0.5              # settling rule of notebook 09

# === copied from leszek-setup.ipynb section 4 (commit adb148b), NOT edited ===
ROOM, T60, C_SOUND, SRC, MIC = [5, 10, 6], 0.2, 340, [1, 2.5, 2], [1, 1.5, 1]
ho = rir.generate(c=C_SOUND, fs=FS, r=MIC, s=SRC, L=ROOM, reverberation_time=T60, nsample=M).flatten()
ho = ho/np.linalg.norm(ho)
lags = np.abs(np.subtract.outer(np.arange(M), np.arange(M)))


def noise_of(ar_a, beta=BETA):
    # Noise scale and b_eta = E|eta| at 5 dB for this input: notebook 09's lines, AR as a parameter.
    P_signal = ho @ (ar_a**lags if ar_a != 0 else np.eye(M)) @ ho
    var_eta = P_signal/10**(SNR_DB/10)
    scale_gg = np.sqrt(var_eta/np.exp(gammaln(3/beta) - gammaln(1/beta)))
    return scale_gg, scale_gg*np.exp(gammaln(2/beta) - gammaln(1/beta))


def signals(seed, ar_a, n, change_at=None, beta=BETA):
    # generate_signals / generate_signals_change of notebook 09 (commit 6e6dabc), with the AR
    # coefficient, the length and the instant of the flip as parameters. Same draws, same order.
    scale_gg, _ = noise_of(ar_a, beta)
    rng = np.random.default_rng(seed)
    u = np.sqrt(1 - ar_a**2)*rng.standard_normal(n + WARMUP)
    x = lfilter([1.0], [1.0, -ar_a], u)[WARMUP:]
    y = np.convolve(ho, x)[:n]
    if change_at is not None:
        y[change_at:] = -y[change_at:]
    eta = gennorm.rvs(beta, scale=scale_gg, size=n, random_state=rng)
    return x, y + eta
# === end of the copied block ===


def noise_variance(ar_a, beta=BETA):
    # The true noise variance, for the Gaussian likelihood: the variance of the generalized Gaussian.
    scale_gg, _ = noise_of(ar_a, beta)
    return scale_gg**2*np.exp(gammaln(3/beta) - gammaln(1/beta))


def stack(sigs):
    return np.array([s[0] for s in sigs]), np.array([s[1] for s in sigs])


for beta, ar_a in ((BETA, -0.9), (BETA, 0.0), (BETA_GAUSS, -0.9), (BETA_GAUSS, 0.0)):
    scale_gg, b_eta = noise_of(ar_a, beta)
    print(f"beta* = {beta:g}, AR {ar_a:>4}: var_eta = {noise_variance(ar_a, beta):.4e}, "
          f"noise scale = {scale_gg:.4e}, "
          f"b_eta = E|eta| = {b_eta:.4e}")
# === end of the copied block ===


def clean_output(X, change_at=None):
    # x_t' theta_t for every row of X, as signals() forms y: theta = h_o, and -h_o from change_at on
    Y = np.array([np.convolve(ho, x)[:X.shape[1]] for x in X])
    if change_at is not None:
        Y[:, change_at:] = -Y[:, change_at:]
    return Y


def signal_power(ar_a):
    # h_o' R_xx h_o: the line of noise_of
    return ho @ (ar_a**lags if ar_a != 0 else np.eye(M)) @ ho


# Copied from bank-skf-fkf.ipynb section 7 (commit 57bd568).
def scenario_signals(ar_a, beta=BETA):
    return (stack([signals(seed, ar_a, N, beta=beta) for seed in range(R)]),
            stack([signals(seed, ar_a, 2*N, change_at=N, beta=beta) for seed in range(R)]))

## 3. The filters

### 3.1 Ours

Section 3.1 of `awu-gisv-vs-banks.ipynb` (commit `fab8cf0`), nothing edited: `sKF_batch`, `fKF_batch`, `KF_batch` and the
sKF and fKF banks (`bank_batch`, Algorithms 1 and 2). Here `sKF_batch` runs the fast sKF and `bank_batch` the bank of the
two sKFs; the other runs of ours are read from `results/kf-bank-q-sweep/`.

In [ ]:
# === copied from awu-gisv-vs-banks.ipynb section 3.1 (commit fab8cf0), NOT edited ===
# === copied from kf-bank-q-sweep.ipynb section 3 (commit 56108a2), NOT edited ===
# === copied from bank-kf-excess-error.ipynb section 3 (commit f2ded06), NOT edited; log_density_gaussian added at the end ===
# === IGNACIO: log_mills / chi_laplacian - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/07_skf_conjunto.ipynb, commit 9406077,
#         via bank-skf-fkf.ipynb (commit 57bd568)
def log_mills(z):
    """log R(z), with R(z) = Phi(-z)/phi(z) the Mills ratio, eq. (40). R grows like e^{z^2/2} for
    negative z and overflows, so it is only ever handled through its logarithm."""
    return log_ndtr(-z) + 0.5*z**2 + 0.5*np.log(2*np.pi)


def chi_laplacian(e, sigma, b_eta):
    """Correction chi_t(e_t) and its slope chi'_t(e_t) for Laplacian noise, eqs. (39), (41) and (43)."""
    u = e/sigma                                        # u_t = e_t / sigma_t
    k_t = sigma/b_eta                                  # k_t = sigma_t / b_eta
    tau = sigma**2/b_eta                               # tau_t = sigma_t^2 / b_eta, the largest correction
    log_R_minus = log_mills(k_t - u)                   # log R(k_t - u_t)
    log_R_plus = log_mills(k_t + u)                    # log R(k_t + u_t)
    Lambda = np.tanh((log_R_minus - log_R_plus)/2)     # (R- - R+)/(R- + R+), in (-1, 1)
    chi = tau*Lambda                                   # chi = tau Lambda
    chi_slope = (2*k_t*np.exp(-np.logaddexp(log_R_minus, log_R_plus))   # 2k / (R- + R+)
                 - k_t**2*(1 - Lambda**2))                               # - k^2 (1 - Lambda^2)
    return chi, chi_slope
# === end of the copied block ===


# Copied from bank-skf-fkf.ipynb (commit 57bd568), from vkf-kf.ipynb.
def chi_gaussian(e, sigma, v_eta):
    """Eq. (38), the Gaussian instance. Used only for the Kalman cross-check of section 4."""
    frac = sigma**2/(sigma**2 + v_eta)
    return frac*e, frac


# sKF_batch and fKF_batch: copied from bank-skf-fkf.ipynb (commit 57bd568), where they carry the options below;
# one option added here, `excess`, and nothing else changed.
#   flip_at: misalignment against -h from that step on (fkf.ipynb had it for the fKF only)
#   clip:    chi' -> max(chi', 0) in the variance update, eq. (36); the mean update is untouched
#   record:  also return chi'_t and v_t at every step
#   v0:      initial v (VAR_THETA_0 = 2 in every notebook of ours; 1/M in Leszek's ggbench.py)
#   wait:    first step that updates (M: wait for a full window, as every notebook of ours)
#   excess:  also return s_t = x_t'(theta_t - w_{t-1}) at every step (not with record)
def _roll_in(X_t, x_win):
    x_win = np.roll(x_win, 1, axis=1)
    x_win[:, 0] = X_t
    return x_win


def sKF_batch(X, D, h, b_eta, epsilon, chi_fn, flip_at=None, clip=False, record=False,
              v0=VAR_THETA_0, wait=None, excess=False):
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    v = np.full(B, float(v0))
    eps = np.asarray(epsilon, dtype=float)
    mis = np.empty((B, n))
    if excess:
        s = np.empty((B, n))
    if record:
        slope_hist, v_hist = np.full((B, n), np.nan), np.full((B, n), np.nan)
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = (w - h) if flip_at is None or t < flip_at else (w + h)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if excess:
            s[:, t] = -np.einsum("bm,bm->b", x_t, dw)          # s_t = x_t'(theta_t - w_{t-1})
        if t < first:
            continue
        v_tilde = v + eps
        power = np.einsum("bm,bm->b", x_t, x_t)
        sigma = np.sqrt(v_tilde*power)
        chi, slope = chi_fn(e, sigma, b_eta)
        w = w + x_t*(chi/power)[:, None]
        v = v_tilde*(1 - (np.maximum(slope, 0.0) if clip else slope)/L)
        if record:
            slope_hist[:, t], v_hist[:, t] = slope, v
    if excess:
        return mis, s
    if record:
        return mis, slope_hist, v_hist
    return mis


def fKF_batch(X, D, h, b_eta, v, chi_fn, flip_at=None, wait=None, excess=False):
    """B fKFs of eq. (37) side by side. Misalignment against h, or against -h from step flip_at on."""
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    v = np.asarray(v, dtype=float)
    mis = np.empty((B, n))
    if excess:
        s = np.empty((B, n))
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = (w - h) if flip_at is None or t < flip_at else (w + h)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if excess:
            s[:, t] = -np.einsum("bm,bm->b", x_t, dw)          # s_t = x_t'(theta_t - w_{t-1})
        if t < first:
            continue
        power = np.einsum("bm,bm->b", x_t, x_t)
        chi, _ = chi_fn(e, np.sqrt(v*power), b_eta)
        w = w + x_t*(chi/power)[:, None]
    if excess:
        return mis, s
    return mis


# KF_batch: copied from vkf-kf.ipynb (commit 058a9f9); options flip_at and excess added (as in sKF_batch above),
# nothing else changed.
def KF_batch(X, D, h, b_eta, epsilon, chi_fn=chi_laplacian, flip_at=None, excess=False):
    L, (B, n) = len(h), X.shape
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    Sigma = np.broadcast_to(np.eye(L)*float(VAR_THETA_0), (B, L, L)).copy()
    eps_I = np.asarray(epsilon, dtype=float)[:, None, None]*np.eye(L)
    mis, k_hist = np.empty((B, n)), np.zeros((B, n))
    if excess:
        s = np.empty((B, n))
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = (w - h) if flip_at is None or t < flip_at else (w + h)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if excess:
            s[:, t] = -np.einsum("bm,bm->b", x_t, dw)          # s_t = x_t'(theta_t - w_{t-1})
        if t < L:
            continue
        St = Sigma + eps_I
        Sx = np.einsum("bij,bj->bi", St, x_t)
        sigma2 = np.einsum("bi,bi->b", x_t, Sx)
        sigma = np.sqrt(sigma2)
        k_hist[:, t] = sigma/b_eta
        chi, slope = chi_fn(e, sigma, b_eta)
        w = w + Sx*(chi/sigma2)[:, None]
        Sigma = St - (slope/sigma2)[:, None, None]*np.einsum("bi,bj->bij", Sx, Sx)
        Sigma = 0.5*(Sigma + np.swapaxes(Sigma, 1, 2))
    if excess:
        return mis, k_hist, s
    return mis, k_hist


# === copied from bank-skf-fkf.ipynb section 4 (commit 57bd568); bank_batch has the option excess added ===
LOG_2PI = np.log(2*np.pi)


def transition_matrix(K, q):
    """p_ij of eq. (eps.chain): q to each neighbour on the grid, 1 - q n_i to stay."""
    P = np.zeros((K, K))
    for i in range(K):
        neighbours = [j for j in (i - 1, i + 1) if 0 <= j < K]   # n_i = 1 at the ends, 2 inside
        P[i, neighbours] = q
        P[i, i] = 1 - q*len(neighbours)
    return P


def log_density_laplacian(e, sigma, b_eta):
    """log p_t(e) of eq. (bank.weights), from eq. (NL.density): log phi(u) + log(R(k-u) + R(k+u)) - log 2b."""
    u = e/sigma                                        # u = e / sigma
    k_t = sigma/b_eta                                  # k = sigma / b_eta
    return (-0.5*u**2 - 0.5*LOG_2PI                    # log phi(u)
            + np.logaddexp(log_mills(k_t - u), log_mills(k_t + u))   # log(R(k - u) + R(k + u))
            - np.log(2*b_eta))


def bank_step(W, V, log_mu, x_t, d_t, power, grid, log_P, chi_fn, log_density, scale):
    """One step of Algorithm 1 (V given) or Algorithm 2 (V = None) for B banks side by side.
    W (K, B, L) locations, V (K, B) variances, log_mu (K, B) log weights, x_t (B, L), d_t and power (B,).
    Returns the new W, V, log_mu, and log c, log p and u = e/sigma of this step (K, B)."""
    L = W.shape[-1]
    # mixing, eqs. (bank.mixing.prob) and (bank.mixing)
    log_joint = log_P[:, :, None] + log_mu[:, None, :]   # log p_ij mu_{t-1,i}, (i, j, B)
    log_c = logsumexp(log_joint, axis=0)                 # log c_{t,j}
    mix = np.exp(log_joint - log_c[None])                # mu_{t-1,i|j}
    W_bar = np.einsum("ijb,ibm->jbm", mix, W)            # w_bar_{t,j}
    if V is not None:
        diff = W[:, None] - W_bar[None]                  # w_{t-1,i} - w_bar_{t,j}, (i, j, B, L)
        spread = np.einsum("ijbm,ijbm->ijb", diff, diff)/L
        V_bar = np.einsum("ijb,ijb->jb", mix, V[:, None] + spread)   # v_bar_{t,j}
        v_tilde = V_bar + grid[:, None]                  # v_tilde = v_bar + eps^(j)
    else:
        v_tilde = np.broadcast_to(grid[:, None], log_mu.shape)       # fKF: the fixed v^(j)
    # one sKF / fKF step per filter, eq. (bank.update)
    sigma = np.sqrt(v_tilde*power)                       # sigma_{t,j}
    e = d_t - np.einsum("jbm,bm->jb", W_bar, x_t)        # e_{t,j}, each filter's own error
    chi, slope = chi_fn(e, sigma, scale)
    W = W_bar + x_t*(chi/power)[..., None]
    if V is not None:
        V = v_tilde*(1 - slope/L)
    # weights, eq. (bank.weights), in logs
    log_p = log_density(e, sigma, scale)
    ell = log_c + log_p
    log_mu = ell - logsumexp(ell, axis=0)
    return W, V, log_mu, log_c, log_p, e/sigma


def bank_batch(X, D, h, grid, q, family, chi_fn, log_density, scale, v0=VAR_THETA_0, flip_at=None,
               wait=None, record=False, clamp=None, excess=False):
    """B banks side by side, one per row of X and D. Returns the misalignment of the output (B, n),
    the weights averaged over the rows (K, n) and the largest |u| = |e|/sigma met. record=True adds,
    per filter and step, log mu, log c, log p and the misalignment of each filter (K, B, n), and the
    output after each step (B, n, L). excess=True adds s_t = x_t'(theta_t - w_hat_{t-1}) (B, n)."""
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    grid = np.asarray(grid, dtype=float)
    K = len(grid)
    with np.errstate(divide="ignore"):
        log_P = np.log(transition_matrix(K, q))          # log 0 = -inf where p_ij = 0
    W = np.zeros((K, B, L))
    V = np.full((K, B), float(v0)) if family == "sKF" else None
    log_mu = np.full((K, B), -np.log(K))                 # mu_0 = 1/K
    x_t, w_hat = np.zeros((B, L)), np.zeros((B, L))
    mis = np.empty((B, n))
    if excess:
        s = np.empty((B, n))
    mu_mean = np.empty((K, n))
    u_max = 0.0
    if record:
        rec = {key: np.full((K, B, n), np.nan) for key in ("log_mu", "log_c", "log_p", "mis_j")}
        rec["w_hat"] = np.full((B, n, L), np.nan)
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        target = h if flip_at is None or t < flip_at else -h
        dw = w_hat - target
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if excess:
            s[:, t] = -np.einsum("bm,bm->b", x_t, dw)          # s_t = x_t'(theta_t - w_hat_{t-1})
        if record:
            dW = W - target
            rec["mis_j"][:, :, t] = np.einsum("jbm,jbm->jb", dW, dW)
        if t >= first:
            power = np.einsum("bm,bm->b", x_t, x_t)
            W, V, log_mu, log_c, log_p, u = bank_step(W, V, log_mu, x_t, D[:, t], power, grid, log_P,
                                                      chi_fn, log_density, scale)
            if clamp is not None:                        # Leszek's mu = max(mu, clamp), renormalized
                log_mu = np.maximum(log_mu - log_mu.max(axis=0), np.log(clamp))
                log_mu = log_mu - logsumexp(log_mu, axis=0)
            w_hat = np.einsum("jb,jbm->bm", np.exp(log_mu), W)          # eq. (bank.output)
            u_max = max(u_max, float(np.abs(u).max()))
            if record:
                rec["log_c"][:, :, t], rec["log_p"][:, :, t], rec["w_hat"][:, t] = log_c, log_p, w_hat
        mu_mean[:, t] = np.exp(log_mu).mean(axis=1)
        if record:
            rec["log_mu"][:, :, t] = log_mu
    out = {"mis": mis, "mu": mu_mean, "u_max": u_max}
    if excess:
        out["s"] = s
    if record:
        out.update(rec)
    return out


EPS_BANK = np.geomspace(1e-9, 1e-4, 6)     # the draft's remark, line 1614
V_BANK = np.geomspace(1e-6, 1e-2, 9)       # Leszek's fkfbank*.py
Q_BANK = 1e-4
# === end of the copied block ===

# Copied from bank-skf-fkf.ipynb section 4 (commit 57bd568), NOT edited.
def log_density_gaussian(e, sigma, v_eta):
    """log p_t(e) for Gaussian noise: N(0, sigma^2 + v_eta), eq. (bank.weights)."""
    s2 = sigma**2 + v_eta
    return -0.5*(LOG_2PI + np.log(s2)) - e**2/(2*s2)


print("filters ready")

### 3.2 The convex combination of two sKFs

Filter 1 is the fast sKF, filter 2 the slow one. Each takes the step of `sKF_batch` with $\chi$ of eq. (39) on its own
error $y_t - \hat y_{t,k}$, $\hat y_{t,k} = \boldsymbol{x}_t^{\mathsf T}\boldsymbol{w}_{t-1,k}$, and keeps its own $v$. The
review's eqs. (3), (4), (15) and its Table 4 (cvx-PN-LMS), in our notation ($y_t$ is the observation, the review's $d(n)$):

$$\hat{\boldsymbol w}_{t-1} = \lambda_t\boldsymbol{w}_{t-1,1} + (1 - \lambda_t)\boldsymbol{w}_{t-1,2}, \qquad
\lambda_t = \mathrm{sgm}(a_t) = \frac{1}{1 + e^{-a_t}}, \qquad e_t = y_t - \boldsymbol{x}_t^{\mathsf T}\hat{\boldsymbol w}_{t-1},$$

$$p_t = \eta\,p_{t-1} + (1 - \eta)(\hat y_{t,1} - \hat y_{t,2})^2, \qquad
a_{t+1} = \Big[a_t + \frac{\mu_a}{p_t}\,\lambda_t(1 - \lambda_t)\,e_t\,(\hat y_{t,1} - \hat y_{t,2})\Big]_{-a^+}^{a^+}.$$

The misalignment is that of $\hat{\boldsymbol w}_{t-1}$, the vector that predicts $y_t$, as for the banks. Then, after
both filters' step, one of the sharing schemes of the review's Table 5, with this step's $\lambda_t$:

| scheme | rule | parameter |
|---|---|---|
| none | | |
| gradual transfer [47] | if $\lambda_t \ge \lambda_0$: $\boldsymbol{w}_{t,2} \leftarrow \ell\,\boldsymbol{w}_{t,2} + (1 - \ell)\,\boldsymbol{w}_{t,1}$ | $\ell$ |
| simple copy [48] | if $\lambda_t \ge \lambda_0$ and $t$ is a multiple of $N_0$: $\boldsymbol{w}_{t,2} \leftarrow \boldsymbol{w}_{t,1}$ | $N_0$ |
| feedback [49] | if $t$ is a multiple of $N_0$: $\boldsymbol{w}_{t,1}, \boldsymbol{w}_{t,2} \leftarrow \lambda_t\boldsymbol{w}_{t,1} + (1 - \lambda_t)\boldsymbol{w}_{t,2}$ | $N_0$ |

$p_t = 0$ only at the first update, where both filters are still at zero and $\hat y_{t,1} = \hat y_{t,2} = 0$; $a$ is
not moved there. `cvx_reference` writes the same equations for one realisation with scalar loops, for the checks.

In [ ]:
# === new: the convex combination of a fast and a slow sKF, cvx-PN-LMS (Table 4 of the review) with the sharing
#     schemes of its Table 5 ===
EPS_FAST = 1e-4            # the top of the sKF bank's grid (EPS_BANK), the fast filter
A_MAX = 4.0                # a+: a_t is kept in [-a+, a+], so lambda_t in [sgm(-4), sgm(4)] = [0.018, 0.982]
ETA = 0.9                  # forgetting factor of the power estimate p_t (the review's Fig. 5)
LAM0 = 0.98                # default threshold of gradual transfer and simple copy (Table 5: 0 << lambda0 < 1), below sgm(4)


def sgm(a):
    return 1/(1 + np.exp(-a))


def cvx_batch(X, D, h, b_eta, eps_pair, mu_a, sharing="none", par=None, lam0=LAM0, flip_at=None, v0=VAR_THETA_0,
              wait=None, excess=False, record=False):
    """B convex combinations of a fast and a slow sKF side by side, one per row of X and D; eps_pair = (fast, slow).
    Filter 1 is the fast sKF, filter 2 the slow one. Each takes the step of sKF_batch with chi_laplacian on its own
    error y_t - x_t' w_{t-1,k}, and keeps its own v. The output is w_hat = lambda_t w_1 + (1 - lambda_t) w_2 (eq. 3),
    lambda_t = sgm(a_t) (eq. 15); a_t follows cvx-PN-LMS (Table 4) on the output error e_t = y_t - x_t' w_hat (eq. 4).
    Sharing (Table 5), after both filters' step, with this step's lambda_t:
      "none"
      "gradual"   par = ell:  if lambda_t >= lam0:                w_2 <- ell w_2 + (1 - ell) w_1
      "copy"      par = N0:   if lambda_t >= lam0 and t % N0 = 0: w_2 <- w_1
      "feedback"  par = N0:   if t % N0 = 0:                      w_1, w_2 <- lambda_t w_1 + (1 - lambda_t) w_2
    Returns a dict: the misalignment of the output "mis" (B, n), lambda_t averaged over the rows "lam" (n,), the share of
    the updating steps that shared "shared" (B,); excess=True adds s_t = x_t'(theta_t - w_hat) "s" (B, n); record=True
    adds the misalignment of each filter "mis_k" (2, B, n) and lambda_t "lam_all" (B, n)."""
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    W = np.zeros((2, B, L))                              # W[0] = w_1 (fast), W[1] = w_2 (slow)
    V = np.full((2, B), float(v0))
    eps = (float(eps_pair[0]), float(eps_pair[1]))
    a, p = np.zeros(B), np.zeros(B)                      # a_0 = 0, so lambda = 1/2 until the first update; p_0 = 0
    x_t = np.zeros((B, L))
    mis, lam_mean, shared = np.empty((B, n)), np.empty(n), np.zeros(B)
    if excess:
        s = np.empty((B, n))
    if record:
        mis_k, lam_all = np.empty((2, B, n)), np.empty((B, n))
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        lam = sgm(a)                                     # lambda_t
        w_hat = lam[:, None]*W[0] + (1 - lam)[:, None]*W[1]          # eq. (3)
        target = h if flip_at is None or t < flip_at else -h
        dw = w_hat - target
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        lam_mean[t] = lam.mean()
        if excess:
            s[:, t] = -np.einsum("bm,bm->b", x_t, dw)          # s_t = x_t'(theta_t - w_hat)
        if record:
            lam_all[:, t] = lam
            for k in range(2):
                dk = W[k] - target
                mis_k[k, :, t] = np.einsum("bm,bm->b", dk, dk)
        if t < first:
            continue
        y = [np.einsum("bm,bm->b", x_t, W[k]) for k in range(2)]   # y_hat_k = x_t' w_{t-1,k}
        e = D[:, t] - (lam*y[0] + (1 - lam)*y[1])                   # eq. (4), the output error
        power = np.einsum("bm,bm->b", x_t, x_t)
        for k in range(2):                                          # the step of sKF_batch, each on its own error
            e_k = D[:, t] - y[k]
            v_tilde = V[k] + eps[k]
            sigma = np.sqrt(v_tilde*power)
            chi, slope = chi_laplacian(e_k, sigma, b_eta)
            W[k] = W[k] + x_t*(chi/power)[:, None]
            V[k] = v_tilde*(1 - slope/L)
        diff = y[0] - y[1]                                          # cvx-PN-LMS, Table 4
        p = ETA*p + (1 - ETA)*diff**2
        safe = p > 0                                                # p = 0 only while y_hat_1 = y_hat_2 = 0
        a = a + np.where(safe, mu_a*lam*(1 - lam)*e*diff/np.where(safe, p, 1.0), 0.0)
        a = np.clip(a, -A_MAX, A_MAX)
        if sharing == "gradual":                                    # Table 5
            on = lam >= lam0
            W[1, on] = par*W[1, on] + (1 - par)*W[0, on]
        elif sharing == "copy":
            on = (lam >= lam0) & (t % par == 0)
            W[1, on] = W[0, on]
        elif sharing == "feedback":
            on = np.full(B, t % par == 0)
            if on[0]:
                W[0] = W[1] = lam[:, None]*W[0] + (1 - lam)[:, None]*W[1]
        else:
            on = np.zeros(B, dtype=bool)
        shared += on
    out = {"mis": mis, "lam": lam_mean, "shared": shared/max(n - first, 1)}
    if excess:
        out["s"] = s
    if record:
        out.update(mis_k=mis_k, lam_all=lam_all)
    return out


def cvx_reference(x, d, L, b_eta, eps_pair, mu_a, sharing="none", par=None, lam0=LAM0, v0=VAR_THETA_0):
    """Table 4 (cvx-PN-LMS) and Table 5 written line by line for one realisation, scalar loops, for the checks only.
    Returns the output w_hat before every step (n, L) and lambda_t (n,)."""
    n = len(x)
    w1, w2 = np.zeros(L), np.zeros(L)
    v1 = v2 = float(v0)
    a, p = 0.0, 0.0
    xw = np.zeros(L)
    w_hist, lam_hist = np.zeros((n, L)), np.zeros(n)
    for t in range(n):
        xw = np.concatenate(([x[t]], xw[:-1]))
        lam = 1/(1 + np.exp(-a))                                   # eq. (15)
        w_hist[t], lam_hist[t] = lam*w1 + (1 - lam)*w2, lam        # eq. (3)
        if t < L:
            continue
        y1, y2 = xw @ w1, xw @ w2
        e = d[t] - (lam*y1 + (1 - lam)*y2)                         # eq. (4)
        power = xw @ xw
        new = []
        for w_k, v_k, eps_k, y_k in ((w1, v1, eps_pair[0], y1), (w2, v2, eps_pair[1], y2)):   # one sKF step each
            v_tilde = v_k + eps_k
            chi, slope = chi_laplacian(d[t] - y_k, np.sqrt(v_tilde*power), b_eta)
            new.append((w_k + xw*chi/power, v_tilde*(1 - slope/L)))
        (w1, v1), (w2, v2) = new
        p = ETA*p + (1 - ETA)*(y1 - y2)**2                         # Table 4, cvx-PN-LMS
        if p > 0:
            a = a + mu_a/p*lam*(1 - lam)*e*(y1 - y2)
        a = min(max(a, -A_MAX), A_MAX)
        if sharing == "gradual" and lam >= lam0:                   # Table 5
            w2 = par*w2 + (1 - par)*w1
        elif sharing == "copy" and lam >= lam0 and t % par == 0:
            w2 = w1.copy()
        elif sharing == "feedback" and t % par == 0:
            w1 = w2 = lam*w1 + (1 - lam)*w2
    return w_hist, lam_hist


print("the convex combination ready")

## 4. Checks

All must pass before any result. Realisations 0-2 of AR($-0.9$), $\beta^* = 0.2$, 3000 steps, the flip at 1500; the two
sKFs of the main scenario.

### 4.1 Inside the combination, each sKF is `sKF_batch`; a scheme that never fires is "none"

Without sharing, the two filters must be `sKF_batch` at their $\varepsilon$, bit for bit (the same lines in the same
order). With $\ell = 1$, or $N_0$ longer than the run, the sharing schemes change nothing and must give "none" bit for
bit.

In [ ]:
X_c4, D_c4 = stack([signals(seed, -0.9, 3000, change_at=1500) for seed in range(3)])
b_c4 = noise_of(-0.9)[1]
PAIR_C4 = (EPS_FAST, 1.3493e-6)              # the slow sKF of section 6 (TUNED_HEAVY, defined in section 5)
none_c4 = cvx_batch(X_c4, D_c4, ho, b_c4, PAIR_C4, 1.0, flip_at=1500, record=True)
for k, name in enumerate(("fast", "slow")):
    ref = sKF_batch(X_c4, D_c4, ho, b_c4, np.full(3, PAIR_C4[k]), chi_laplacian, flip_at=1500)
    print(f"{name} sKF inside the combination against sKF_batch: largest |difference| of the misalignment "
          f"{np.max(np.abs(none_c4['mis_k'][k] - ref)):.1e}")
for sharing, par in (("gradual", 1.0), ("copy", 3001), ("feedback", 3001)):
    other = cvx_batch(X_c4, D_c4, ho, b_c4, PAIR_C4, 1.0, sharing, par, flip_at=1500)
    print(f"{sharing:<9} at {par:g}, against none: largest |difference| of the output's misalignment "
          f"{np.max(np.abs(other['mis'] - none_c4['mis'])):.1e}")

### 4.2 Against Tables 4 and 5 line by line

`cvx_batch` against `cvx_reference`, each scheme with a setting that fires ($\mu_a = 1$; $\ell = 0.9$, $N_0 = 2$ for the
copy, $N_0 = 10$ for the feedback). The two compute the same sums in another order, and the Laplacian $\chi$ amplifies
rounding (`kf-bank-q-sweep.ipynb`, check 4.1), so the yardstick is printed beside: the reference against itself with
$\boldsymbol{x}_t$ scaled by $1 + 2^{-52}$, one unit in the last place.

$\lambda_t$ is the sensitive part. While $\hat y_{t,1} \approx \hat y_{t,2}$, the step of $a_t$ is about
$\mu_a e_t/((1 - \eta)(\hat y_{t,1} - \hat y_{t,2}))$, which is very large, so $a_t$ jumps between $\pm a^+$ and one bit can
send it to the other side. At those steps the two filters are nearly equal, so the output hardly depends on $\lambda_t$.
Printed: the largest gap of the output's misalignment, the largest gap of $\lambda_t$, and on how many steps (of
$3 \times 3000$) $\lambda_t$ differs by more than $10^{-6}$.

In [ ]:
tgt_c4 = np.where(np.arange(3000)[:, None] < 1500, ho, -ho)
print(f"{'':<10}{'cvx_batch against the reference':-^42}  {'the reference against itself, one ulp':-^42}")
print(f"{'sharing':<10}{'misalignment':>14}{'lambda':>10}{'steps':>8}{'shared':>10}  {'misalignment':>14}{'lambda':>10}"
      f"{'steps':>8}")
for sharing, par in (("none", None), ("gradual", 0.9), ("copy", 2), ("feedback", 10)):
    fast = cvx_batch(X_c4, D_c4, ho, b_c4, PAIR_C4, 1.0, sharing, par, flip_at=1500, record=True)
    gap, ulp = np.zeros(3), np.zeros(3)
    n_gap = n_ulp = 0
    for r in range(3):
        w_ref, lam_ref = cvx_reference(X_c4[r], D_c4[r], M, b_c4, PAIR_C4, 1.0, sharing, par)
        w_ulp, lam_ulp = cvx_reference(X_c4[r]*(1 + 2.0**-52), D_c4[r], M, b_c4, PAIR_C4, 1.0, sharing, par)
        mis_ref, mis_ulp = ((w_ref - tgt_c4)**2).sum(axis=1), ((w_ulp - tgt_c4)**2).sum(axis=1)
        gap = np.maximum(gap, [np.max(np.abs(fast["mis"][r] - mis_ref)/mis_ref),
                               np.max(np.abs(fast["lam_all"][r] - lam_ref)), 0])
        ulp = np.maximum(ulp, [np.max(np.abs(mis_ulp - mis_ref)/mis_ref), np.max(np.abs(lam_ulp - lam_ref)), 0])
        n_gap += int((np.abs(fast["lam_all"][r] - lam_ref) > 1e-6).sum())
        n_ulp += int((np.abs(lam_ulp - lam_ref) > 1e-6).sum())
    print(f"{sharing:<10}{gap[0]:>14.1e}{gap[1]:>10.2f}{n_gap:>8d}{fast['shared'].mean():>10.3f}  {ulp[0]:>14.1e}"
          f"{ulp[1]:>10.2f}{n_ulp:>8d}")
print("   misalignment: largest relative difference; lambda: largest |difference|; steps: |lambda difference| > 1e-6;\n"
      "   shared: share of the updating steps on which the scheme fired (mean over the 3 rows)")

### 4.3 The bank of the two sKFs at $q = 0$

With $q = 0$ the two filters of the bank exchange nothing and must be `sKF_batch` at their $\varepsilon$, bit for bit, as
the bank of six was in `bank-skf-fkf.ipynb`.

In [ ]:
bank_c4 = bank_batch(X_c4, D_c4, ho, [PAIR_C4[1], PAIR_C4[0]], 0.0, "sKF", chi_laplacian, log_density_laplacian, b_c4,
                     flip_at=1500, record=True)
for j, name in enumerate(("slow", "fast")):
    ref = sKF_batch(X_c4, D_c4, ho, b_c4, np.full(3, PAIR_C4[1 - j]), chi_laplacian, flip_at=1500)
    print(f"bank, q = 0, {name} sKF against sKF_batch: largest |difference| of the misalignment "
          f"{np.nanmax(np.abs(bank_c4['mis_j'][j] - ref)):.1e}")
del X_c4, D_c4, none_c4, bank_c4

## 5. Measurement

Section 5 of `awu-gisv-vs-banks.ipynb` (commit `fab8cf0`), nothing edited: notebook 09's measures, `summarise`, `cached`,
`tuned_values` and `filter_run`, which read the runs of ours from `results/kf-bank-q-sweep/` (and recompute one only if
its file is missing).

New below them:
- `new_run`: a filter of this notebook, rest run and change run, summarised by `summarise`; saved under
  `results/cvx-combination-vs-bank/`.
- `scenario_combination`: one scenario. The control (ours read from the cache against the numbers printed by
  `kf-bank-q-sweep.ipynb`), the fast sKF, the bank of the two sKFs at every $q$, the combination with every scheme at every
  setting, then three tables: every setting's mean over the record, and the filters at their best setting from the mean
  and from the median curves.

In [ ]:
# === copied from awu-gisv-vs-banks.ipynb section 5 (commit fab8cf0), NOT edited ===
# === copied from kf-bank-q-sweep.ipynb section 5 (commit 56108a2), NOT edited ===
# === copied from bank-kf-excess-error.ipynb section 5 (commit f2ded06), NOT edited ===
# === IGNACIO: steady_state, is_settled and recovery - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/09_matcheado.ipynb, commit 6e6dabc,
#         via bank-skf-fkf.ipynb (commit 57bd568)
def steady_state(misalignment):
    """Floor in dB, and the first step within 3 dB of it."""
    tail = slice(3*len(misalignment)//4, len(misalignment))
    floor = 10*np.log10(misalignment[tail].mean())
    db = 10*np.log10(misalignment)
    reached = int(np.argmax(db < floor + 3))
    return floor, reached


def is_settled(misalignment, drift):
    """Settled within the run: within 3 dB of the floor by half of it, and no longer descending."""
    n = len(misalignment)
    floor, reached = steady_state(misalignment)
    third_quarter = 10*np.log10(misalignment[n//2:3*n//4].mean())
    # reached = 0: the run never left the 3 dB band around its start, so it never converged either
    return bool(0 < reached <= n/2 and third_quarter - floor <= drift)


def recovery(misalignment, floor):
    """Steps from the change until the misalignment is back within 3 dB of the floor at rest;
    -1 if it never gets there before the run ends."""
    db = 10*np.log10(misalignment[N:])
    back = db < floor + 3
    if not back.any():
        return -1
    return int(np.argmax(back))
# === end of the copied block ===


# === copied from bank-skf-fkf.ipynb sections 7 and 7.2 (commit 57bd568), NOT edited ===
def time_to(misalignment, level_db):
    """First step after the change below level_db; -1 if never."""
    below = 10*np.log10(misalignment[N:]) < level_db
    return int(np.argmax(below)) if below.any() else -1


LEVELS = [0.0, -5.0, -10.0, -15.0]


def first_below(misalignment, level_db):
    below = 10*np.log10(misalignment) < level_db
    return int(np.argmax(below)) if below.any() else -1


def measure(rest_curve, change_curve):
    floor, reached = steady_state(rest_curve)
    return dict(floor=floor, reached=reached, settled=is_settled(rest_curve, DRIFT_DB),
                to_17=first_below(rest_curve, TARGET_DB + 3), to_20=first_below(rest_curve, TARGET_DB),
                levels=[time_to(change_curve, level) for level in LEVELS], rec=recovery(change_curve, floor),
                back_20=time_to(change_curve, TARGET_DB), mean_db=10*np.log10(change_curve.mean()),
                curve=change_curve)


def show_table(rows):
    print(f"{'':<34}{'at rest':-^46}  {'after the flip':-^50}")
    print(f"{'filter':<22}{'eps or v':>12}{'floor':>8}{'settled':>9}{'own+3':>7}{'-17 dB':>8}{'-20 dB':>8}"
          + "".join(f"{level:>8.0f}" for level in LEVELS) + f"{'own+3':>9}{'-20 dB':>9}{'mean [dB]':>11}")
    for name, r in rows.items():
        value = "grid" if np.isnan(r["value"]) else f"{r['value']:.4e}"
        print(f"{name:<22}{value:>12}{r['floor']:>8.2f}{'yes' if r['settled'] else 'NO':>9}{r['reached']:>7d}"
              f"{r['to_17']:>8d}{r['to_20']:>8d}" + "".join(f"{s:>8d}" for s in r["levels"])
              + f"{r['rec']:>9d}{r['back_20']:>9d}{r['mean_db']:>11.2f}")
    print("   steps; -1 = never within the run")


EPS_GRID = np.logspace(-8, -3, 11)          # notebook 09


def pick(grid, floors, settled, target):
    values, kept = grid[settled], floors[settled]
    if target < kept.min():
        return values[np.argmin(kept)], "target below every settled floor"
    if target > kept.max():
        return values[np.argmax(kept)], "target above every settled floor"
    order = np.argsort(kept)
    return 10**np.interp(target, kept[order], np.log10(values)[order]), "ok"
# === end of the copied block ===


def sweep_kf(b_eta, X, D):
    # notebook 09's search for the KF, as sweep() of the bank notebook, one grid value at a time
    floors, settled = [], []
    for eps in EPS_GRID:
        curve = KF_batch(X, D, ho, b_eta, np.full(len(X), eps))[0].mean(axis=0)
        floors.append(steady_state(curve)[0])
        settled.append(is_settled(curve, DRIFT_DB))
    return np.array(floors), np.array(settled)


SMOOTH = 101                # centred moving average on both means, in steps


def smoothed_ratio(num, den, cut=None):
    """Moving average of both means, then the ratio. No window crosses step cut (the change)."""
    if cut is not None:
        return np.concatenate([smoothed_ratio(num[:cut], den[:cut]), smoothed_ratio(num[cut:], den[cut:])])
    return uniform_filter1d(num, SMOOTH, mode="nearest")/uniform_filter1d(den, SMOOTH, mode="nearest")


def measure_excess(s_rest, s_change, den_rest, den_change):
    """s_rest (R, N), s_change (R, 2N); den_* = E[(x_t' theta_t)^2], the mean over realisations at each step."""
    num_rest, num_change = (s_rest**2).mean(axis=0), (s_change**2).mean(axis=0)
    rest_curve, change_curve = smoothed_ratio(num_rest, den_rest), smoothed_ratio(num_change, den_change, cut=N)
    tail = slice(3*N//4, N)                            # the window of steady_state
    return dict(floor=10*np.log10(num_rest[tail].mean()/den_rest[tail].mean()),
                to_17=first_below(rest_curve, TARGET_DB + 3), to_20=first_below(rest_curve, TARGET_DB),
                levels=[time_to(change_curve, level) for level in LEVELS + [TARGET_DB]],
                mean_db=10*np.log10(num_change.mean()/den_change.mean()),
                rest_curve=rest_curve, curve=change_curve)


def show_excess(rows):
    print(f"{'':<34}{'at rest':-^37}  {'after the flip, excess error':-^40}")
    print(f"{'filter':<22}{'eps or v':>12}{'mis.':>8}{'excess':>8}{'gap':>7}{'-17 dB':>7}{'-20 dB':>7}"
          + "".join(f"{level:>8.0f}" for level in LEVELS + [TARGET_DB]) + f"{'mean mis.':>11}{'mean exc.':>11}")
    for name, r in rows.items():
        x = r["excess"]
        value = "grid" if np.isnan(r["value"]) else f"{r['value']:.4e}"
        print(f"{name:<22}{value:>12}{r['floor']:>8.2f}{x['floor']:>8.2f}{x['floor'] - r['floor']:>7.2f}"
              f"{x['to_17']:>7d}{x['to_20']:>7d}" + "".join(f"{s:>8d}" for s in x["levels"])
              + f"{r['mean_db']:>11.2f}{x['mean_db']:>11.2f}")
    print("   floors and means in dB; mis. = misalignment, excess = excess error, gap = excess - mis.")
    print("   steps on the excess error (smoothed, good to about 50 steps); -1 = never within the run")


def run_filter(name, value, b_eta, X, D, flip_at=None):
    # misalignment and s_t, (B, n) each
    family, method = name.split(", ")
    if method == "bank":
        out = bank_batch(X, D, ho, EPS_BANK if family == "sKF" else V_BANK, Q_BANK, family, chi_laplacian,
                         log_density_laplacian, b_eta, flip_at=flip_at, excess=True)
        return out["mis"], out["s"]
    values = np.full(len(X), value)
    if family == "sKF":
        return sKF_batch(X, D, ho, b_eta, values, chi_laplacian, flip_at=flip_at, excess=True)
    if family == "fKF":
        return fKF_batch(X, D, ho, b_eta, values, chi_laplacian, flip_at=flip_at, excess=True)
    mis, _, s = KF_batch(X, D, ho, b_eta, values, chi_laplacian, flip_at=flip_at, excess=True)
    return mis, s


# floor, own + 3, recovery and mean over the record printed by bank-skf-fkf.ipynb (commit 57bd568)
BANK_NB = {(-0.9, "sKF, exact"): (-20.01, 4756, 7584, -14.08), (-0.9, "sKF, bank"): (-25.92, 68043, 75315, -13.89),
           (-0.9, "fKF, exact"): (-19.99, 5461, 8188, -13.45), (-0.9, "fKF, bank"): (-15.68, 28234, 29311, -11.92),
           (0.0, "sKF, exact"): (-20.01, 1564, 1221, -17.22), (0.0, "sKF, bank"): (-44.92, 6963, 6433, -22.64),
           (0.0, "fKF, exact"): (-20.00, 1056, 1596, -16.67), (0.0, "fKF, bank"): (-44.58, 12135, 12233, -23.01)}


def scenario_run(ar_a, tuned):
    """The KF tuned to -20 dB, then the five filters at rest and across the change, with s_t."""
    start = time.time()
    (X_r, D_r), (X_c, D_c) = scenario_signals(ar_a)
    b_eta = noise_of(ar_a)[1]
    den_r = (clean_output(X_r)**2).mean(axis=0)
    den_c = (clean_output(X_c, change_at=N)**2).mean(axis=0)
    print(f"mean of (x_t' theta_t)^2 over the change runs: {den_c[M:].mean():.4f}; "
          f"h' R_xx h = {signal_power(ar_a):.4f}")
    floors, settled = sweep_kf(b_eta, X_r, D_r)
    eps_kf, status = pick(EPS_GRID, floors, settled, TARGET_DB)
    print("KF: grid floors " + " ".join(f"{f:.1f}" + ("" if s else "*") for f, s in zip(floors, settled))
          + "   (* not settled)")
    print(f"   at -20 dB: eps = {eps_kf:.4e}, {status}   [{time.time() - start:.0f} s]")
    values = {"sKF, exact": tuned["sKF"], "sKF, bank": np.nan, "fKF, exact": tuned["fKF"], "fKF, bank": np.nan,
              "KF, exact": eps_kf}
    rows = {}
    for name, value in values.items():
        mis_r, s_r = run_filter(name, value, b_eta, X_r, D_r)
        mis_c, s_c = run_filter(name, value, b_eta, X_c, D_c, flip_at=N)
        rows[name] = dict(value=value, rest_curve=mis_r.mean(axis=0), **measure(mis_r.mean(axis=0), mis_c.mean(axis=0)),
                          excess=measure_excess(s_r, s_c, den_r, den_c))
        print(f"{name} done [{time.time() - start:.0f} s]")
    print("\ncontrol: the four filters of the bank notebook against its printed numbers (ours / printed)")
    for name, r in rows.items():
        if (ar_a, name) in BANK_NB:
            floor, steps, rec, mean = BANK_NB[(ar_a, name)]
            print(f"   {name:<11} floor {r['floor']:.2f} / {floor:.2f}, own+3 {r['reached']} / {steps}, "
                  f"recovery {r['rec']} / {rec}, mean {r['mean_db']:.2f} / {mean:.2f}")
    print("\nmisalignment")
    show_table(rows)
    print("\nexcess error")
    show_excess(rows)
    return rows


print("measurement ready")

In [ ]:
# === copied from awu-gisv-vs-banks.ipynb section 5.1 (commit fab8cf0), NOT edited ===
# === copied from kf-bank-q-sweep.ipynb section 5.1 (commit 56108a2); one option added to cached (folder,
#     default CACHE), nothing else edited ===
Q_SWEEP = [1e-4, 1e-3, 1e-2, 1e-1, 1/3]       # 1e-4 is the value of every earlier bank run
V_GRID = np.logspace(-6, -1, 11)               # fkf.ipynb, leszek-setup.ipynb; the fKF search of bank-skf-fkf.ipynb


# === copied from bank-skf-fkf.ipynb section 7.2 (commit 57bd568), NOT edited ===
def sweep(family, chi_fn, scale, grid, X, D):
    # notebook 09's search, as clipped-matched.ipynb section 5: every grid point at R realisations at once
    g = len(grid)
    Xg, Dg, values = np.repeat(X, g, axis=0), np.repeat(D, g, axis=0), np.tile(grid, len(X))
    mis = (sKF_batch(Xg, Dg, ho, scale, values, chi_fn) if family == "sKF"
           else fKF_batch(Xg, Dg, ho, scale, values, chi_fn))
    curves = mis.reshape(len(X), g, -1).mean(axis=0)
    del mis
    return (np.array([steady_state(c)[0] for c in curves]),
            np.array([is_settled(c, DRIFT_DB) for c in curves]))
# === end of the copied block ===


# The single filters at beta* = 0.2, tuned to -20 dB in the earlier notebooks and given here, not re-tuned: sKF and fKF
# from bank-skf-fkf.ipynb, the KF from bank-kf-excess-error.ipynb (printed there to five digits).
TUNED_HEAVY = {-0.9: {"sKF": 1.3493e-6, "fKF": 2.985e-4, "KF": 8.8577e-7},
               0.0: {"sKF": 1.319e-6, "fKF": 3.017e-4, "KF": 1.6553e-6}}
# floor, own+3, recovery and mean over the record printed by bank-kf-excess-error.ipynb (commit f2ded06)
PREVIOUS = dict(BANK_NB)
PREVIOUS.update({(-0.9, "KF, exact"): (-20.01, 1254, 1972, -15.63), (0.0, "KF, exact"): (-20.02, 603, 791, -17.65)})

CACHE = Path("results/kf-bank-q-sweep")        # one file per run, written as it finishes; delete it to rerun all
WINDOW = slice(N - 2000, N + 20000)            # the change run kept at full resolution: -0.25 s to +2.5 s
THIN = 4                                       # the rest run kept every THIN steps


def cached(key, compute, folder=CACHE):
    path = folder/f"{key}.pkl"
    if path.exists():
        with open(path, "rb") as f:
            return pickle.load(f)
    result = compute()
    folder.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(result, f, protocol=pickle.HIGHEST_PROTOCOL)
    return result


def tuned_values(ar_a, beta, b_eta, X, D):
    """The single filters' -20 dB values: given at beta* = 0.2, from notebook 09's search at any other beta*
    (the searches of the bank notebook: settled points only, interpolated)."""
    if beta == BETA:
        return dict(TUNED_HEAVY[ar_a])

    def search():
        found = {}
        for family, grid in (("sKF", EPS_GRID), ("fKF", V_GRID), ("KF", EPS_GRID)):
            floors, settled = (sweep_kf(b_eta, X, D) if family == "KF"
                               else sweep(family, chi_laplacian, b_eta, grid, X, D))
            if settled.any():
                value, status = pick(grid, floors, settled, TARGET_DB)
            else:                                      # pick needs one settled point; say so instead of failing
                value, status = pick(grid, floors, np.ones(len(grid), dtype=bool), TARGET_DB)
                status = "NO grid point settled; picked among all of them, " + status
            found[family] = dict(floors=floors, settled=settled, value=value, status=status)
        return found
    found = cached(f"search_ar{ar_a:g}_beta{beta:g}", search)
    for family, r in found.items():
        print(f"{family}: grid floors " + " ".join(f"{f:.1f}" + ("" if s else "*") for f, s in
                                                     zip(r["floors"], r["settled"])) + "   (* not settled)")
        print(f"   at -20 dB: {r['value']:.4e}, {r['status']}")
    return {family: r["value"] for family, r in found.items()}


def run_one(name, value, q, b_eta, X, D, flip_at=None):
    """Misalignment and s_t (B, n) of one filter, and its weights averaged over the rows (K, n) if it is a bank.
    run_filter of bank-kf-excess-error.ipynb, with the banks' q as a parameter and the bank of KFs added."""
    family, method = name.split(", ")
    if method == "bank":
        out = (kf_bank_batch(X, D, ho, EPS_BANK, q, chi_laplacian, log_density_laplacian, b_eta, flip_at=flip_at,
                             excess=True) if family == "KF" else
               bank_batch(X, D, ho, EPS_BANK if family == "sKF" else V_BANK, q, family, chi_laplacian,
                          log_density_laplacian, b_eta, flip_at=flip_at, excess=True))
        return out["mis"], out["s"], out["mu"]
    values = np.full(len(X), value)
    if family == "sKF":
        mis, s = sKF_batch(X, D, ho, b_eta, values, chi_laplacian, flip_at=flip_at, excess=True)
    elif family == "fKF":
        mis, s = fKF_batch(X, D, ho, b_eta, values, chi_laplacian, flip_at=flip_at, excess=True)
    else:
        mis, _, s = KF_batch(X, D, ho, b_eta, values, chi_laplacian, flip_at=flip_at, excess=True)
    return mis, s, None


def smooth_split(a, cut=None):
    """The 101-step centred moving average of smoothed_ratio, along the last axis; no window crosses step cut."""
    if cut is not None:
        return np.concatenate([smooth_split(a[..., :cut]), smooth_split(a[..., cut:])], axis=-1)
    return uniform_filter1d(a, SMOOTH, axis=-1, mode="nearest")


def measure_excess_median(num_rest, num_change, den_rest, den_change):
    """measure_excess for the median: num_* are the medians over the realisations of each realisation's s_t^2,
    already smoothed (the median of 20 raw squares is noisy and reads low); den_* the means of (x_t' theta_t)^2."""
    rest_curve, change_curve = num_rest/smooth_split(den_rest), num_change/smooth_split(den_change, cut=N)
    tail = slice(3*N//4, N)                            # the window of steady_state
    return dict(floor=10*np.log10(num_rest[tail].mean()/den_rest[tail].mean()),
                to_17=first_below(rest_curve, TARGET_DB + 3), to_20=first_below(rest_curve, TARGET_DB),
                levels=[time_to(change_curve, level) for level in LEVELS + [TARGET_DB]],
                mean_db=10*np.log10(num_change.mean()/den_change.mean()),
                rest_curve=rest_curve, curve=change_curve)


def summarise(mis_r, s_r, mis_c, s_c, mu_c, den_r, den_c):
    """Numbers and curves of one filter in one scenario, from its rest run (R, N) and its change run (R, 2N).
    mean: the mean over the realisations at each step, measured as in the earlier notebooks (measure,
    measure_excess). median: the median over the realisations at each step, of the misalignment and of each
    realisation's s_t^2 smoothed over 101 steps, measured by the same functions. The 25 % and 75 % curves are kept
    for the bands of the figures."""
    pct = (25, 50, 75)
    mis_q_r, mis_q_c = np.percentile(mis_r, pct, axis=0), np.percentile(mis_c, pct, axis=0)
    sm_r, sm_c = smooth_split(s_r**2), smooth_split(s_c**2, cut=N)
    exc_q_r, exc_q_c = np.percentile(sm_r, pct, axis=0), np.percentile(sm_c, pct, axis=0)
    mean_r, mean_c = mis_r.mean(axis=0), mis_c.mean(axis=0)
    out = {"mean": measure(mean_r, mean_c), "median": measure(mis_q_r[1], mis_q_c[1]),
           "mean_x": measure_excess(s_r, s_c, den_r, den_c),
           "median_x": measure_excess_median(exc_q_r[1], exc_q_c[1], den_r, den_c)}
    den_sm_r, den_sm_c = smooth_split(den_r), smooth_split(den_c, cut=N)
    f32 = lambda a: np.asarray(a, dtype=np.float32)
    out["curves"] = {
        "mis_mean": f32(mean_c[WINDOW]), "mis_q": f32(mis_q_c[:, WINDOW]),
        "exc_mean": f32(out["mean_x"]["curve"][WINDOW]), "exc_q": f32((exc_q_c/den_sm_c)[:, WINDOW]),
        "rest_mis_mean": f32(mean_r[::THIN]), "rest_mis_q": f32(mis_q_r[:, ::THIN]),
        "rest_exc_mean": f32(out["mean_x"]["rest_curve"][::THIN]), "rest_exc_q": f32((exc_q_r/den_sm_r)[:, ::THIN]),
        "mu": None if mu_c is None else f32(mu_c[:, ::8])}
    for key in ("mean", "median", "mean_x", "median_x"):         # the full-length curves are not kept
        out[key].pop("curve", None)
        out[key].pop("rest_curve", None)
    return out


def filter_run(name, value, q, ar_a, beta, sig, b_eta, den):
    """One filter in one scenario, rest and change runs, summarised; read from the cache when it has been run."""
    tag = "grid" if np.isnan(value) else f"{value:.6e}"
    key = f"{name.replace(', ', '-')}_{tag}_q{'-' if q is None else f'{q:g}'}_ar{ar_a:g}_beta{beta:g}"

    def compute():
        start = time.time()
        (X_r, D_r), (X_c, D_c) = sig
        mis_r, s_r, _ = run_one(name, value, q, b_eta, X_r, D_r)
        mis_c, s_c, mu_c = run_one(name, value, q, b_eta, X_c, D_c, flip_at=N)
        out = summarise(mis_r, s_r, mis_c, s_c, mu_c, *den)
        out.update(name=name, value=value, q=q, ar_a=ar_a, beta=beta, seconds=time.time() - start)
        return out
    return cached(key, compute)


def show_sweep(rows, stat):
    """The numbers of one scenario, from the mean curves (stat = "mean") or the median curves (stat = "median")."""
    print(f"{'':<22}{' misalignment, ' + stat + ' ':-^72}  {' excess error, ' + stat + ' ':-^38}")
    print(f"{'filter':<12}{'q':>10}{'floor':>8}{'sett.':>6}{'own+3':>7}{'-17':>7}{'0':>7}{'-10':>7}{'-15':>7}"
          f"{'rec.':>7}{'-20':>7}{'mean':>8}  {'floor':>7}{'-10':>7}{'-15':>7}{'-20':>7}{'mean':>8}")
    for (name, q), r in rows.items():
        m, x = r[stat], r[stat + "_x"]
        print(f"{name:<12}{'' if q is None else f'{q:.3g}':>10}{m['floor']:>8.2f}{'yes' if m['settled'] else 'NO':>6}"
              f"{m['reached']:>7d}{m['to_17']:>7d}{m['levels'][0]:>7d}{m['levels'][2]:>7d}{m['levels'][3]:>7d}"
              f"{m['rec']:>7d}{m['back_20']:>7d}{m['mean_db']:>8.2f}  {x['floor']:>7.2f}{x['levels'][2]:>7d}"
              f"{x['levels'][3]:>7d}{x['levels'][4]:>7d}{x['mean_db']:>8.2f}")
    print("   floors and means in dB, everything else in steps; own+3 (to the own floor + 3 dB) and -17 from zero at "
          "rest;\n   0, -10, -15, rec. (back to the own floor + 3 dB) and -20 dB after the flip; -1 = never within "
          "the run")


def scenario_sweep(ar_a, beta):
    """The three single filters at -20 dB and the three banks at each q of Q_SWEEP, in one scenario."""
    start = time.time()
    sig = scenario_signals(ar_a, beta)
    (X_r, D_r), (X_c, D_c) = sig
    b_eta = noise_of(ar_a, beta)[1]
    den = ((clean_output(X_r)**2).mean(axis=0), (clean_output(X_c, change_at=N)**2).mean(axis=0))
    print(f"b_eta = E|eta| = {b_eta:.4e}; mean of (x_t' theta_t)^2 over the change runs {den[1][M:].mean():.4f}, "
          f"h' R_xx h = {signal_power(ar_a):.4f}")
    tuned = tuned_values(ar_a, beta, b_eta, X_r, D_r)
    rows = {}
    for family in ("sKF", "fKF", "KF"):
        rows[(f"{family}, exact", None)] = filter_run(f"{family}, exact", tuned[family], None, ar_a, beta, sig, b_eta,
                                                      den)
    for q in Q_SWEEP:
        for family in ("sKF", "fKF", "KF"):
            rows[(f"{family}, bank", q)] = filter_run(f"{family}, bank", np.nan, q, ar_a, beta, sig, b_eta, den)
        print(f"q = {q:.3g} done [{time.time() - start:.0f} s]", flush=True)
    if beta == BETA:
        print("\ncontrol: the filters run before, against their printed numbers (ours / printed)")
        for (name, q), r in rows.items():
            if (ar_a, name) in PREVIOUS and q in (None, Q_BANK):
                floor, steps, rec, mean = PREVIOUS[(ar_a, name)]
                m = r["mean"]
                print(f"   {name:<11} floor {m['floor']:.2f} / {floor:.2f}, own+3 {m['reached']} / {steps}, "
                      f"recovery {m['rec']} / {rec}, mean {m['mean_db']:.2f} / {mean:.2f}")
    print()
    show_sweep(rows, "mean")
    print()
    show_sweep(rows, "median")
    print(f"[{time.time() - start:.0f} s; the 18 runs took {sum(r['seconds'] for r in rows.values())/60:.0f} min when they were run (a run read from results/ keeps its own time)]")
    return rows


results = {}
print("sweep ready")

In [ ]:
NEW_CACHE = Path("results/cvx-combination-vs-bank")    # this notebook's runs, one file each, written as they finish
MU_A_GRID = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0]                # 3 added in the third run (see the introduction)
LAM0_GRID = [0.98, 0.9, 0.7, 0.5]                      # 0.98 alone in the first run, 0.5 added in the third
SHARING = {"none": [(None, None)],                      # (parameter, lambda0) of each setting
           "gradual": [(ell, lam0) for lam0 in LAM0_GRID for ell in (0.5, 0.9, 0.99)],
           "copy": [(n0, lam0) for lam0 in LAM0_GRID for n0 in (2, 10, 100)],
           "feedback": [(n0, None) for n0 in (2, 10, 100)]}
PAR_NAME = {"none": "", "gradual": "ell", "copy": "N0", "feedback": "N0"}
Q_PAIR = [0.0, 1e-4, 1e-3, 1e-2, 1e-1, 1/3]

# The bank of six sKFs' best q with AR(-0.9) input (BEST_Q of awu-gisv-vs-banks.ipynb, checked there against the sweep)
BEST_Q6 = {BETA: 1e-1, BETA_GAUSS: 1e-2}
# The control: means over the record [dB], mean curves, printed by kf-bank-q-sweep.ipynb (its sections 6 and 8)
PRINTED = {BETA: {("sKF, exact", None): -14.08, ("sKF, bank", 1e-4): -13.89, ("sKF, bank", 1e-1): -16.52},
           BETA_GAUSS: {("sKF, exact", None): -6.77, ("sKF, bank", 1e-4): -8.68, ("sKF, bank", 1e-2): -10.53}}


def new_run(key, run, sig, den):
    """A filter of this notebook in one scenario. run(X, D, flip_at) returns a dict with "mis" and "s" (B, n), and
    "mu" (2, n: slow, fast) and "shared" (B,) when it has them. Rest run and change run, summarised by summarise, plus
    the share of the steps that shared (rest, change). Saved under NEW_CACHE."""
    def compute():
        start = time.time()
        (X_r, D_r), (X_c, D_c) = sig
        r, c = run(X_r, D_r, None), run(X_c, D_c, N)
        out = summarise(r["mis"], r["s"], c["mis"], c["s"], c.get("mu"), *den)
        out["shared"] = None if "shared" not in c else (float(r["shared"].mean()), float(c["shared"].mean()))
        out.update(key=key, seconds=time.time() - start)
        return out
    return cached(key, compute, folder=NEW_CACHE)


def setting_of(sharing, mu_a, par, lam0=None):
    return (f"mu_a {mu_a:g}" + ("" if par is None else f", {PAR_NAME[sharing]} {par:g}")
            + ("" if lam0 is None else f", lam0 {lam0:g}"))


def show_settings(sweep, best):
    """Every setting's mean over the record [dB], mean curves; * marks the best of each scheme."""
    print("mean over the record [dB], mean curves, at every setting; * = the best of each scheme")
    print(f"{'':<36}" + "".join(f"{f'mu_a {m:g}':>11}" for m in MU_A_GRID))
    for sharing, pars in SHARING.items():
        for par, lam0 in pars:
            label = f"cvx, {sharing}" + setting_of(sharing, 0, par, lam0)[6:]
            print(f"{label:<36}" + "".join(f"{sweep[sharing][(m, par, lam0)]['mean']['mean_db']:>10.2f}"
                                           f"{'*' if best[sharing] == (m, par, lam0) else ' '}" for m in MU_A_GRID))
    bank = sweep["bank, K = 2"]
    print(f"{'bank of the two, at q':<36}" + "".join(f"{f'{q:.3g}':>11}" for q in Q_PAIR))
    print(f"{'':<36}" + "".join(f"{bank[q]['mean']['mean_db']:>10.2f}{'*' if best['bank, K = 2'] == q else ' '}"
                                for q in Q_PAIR))


def show_rows(rows, stat):
    """show_compare of awu-gisv-vs-banks.ipynb, with this notebook's names and settings, plus the share of the steps
    on which the sharing fired (at rest / in the change run)."""
    print(f"{'':<52}{' misalignment, ' + stat + ' ':-^72}  {' excess error, ' + stat + ' ':-^38}")
    print(f"{'filter':<22}{'setting':>30}{'floor':>8}{'sett.':>6}{'own+3':>7}{'-17':>7}{'0':>7}{'-10':>7}{'-15':>7}"
          f"{'rec.':>7}{'-20':>7}{'mean':>8}  {'floor':>7}{'-10':>7}{'-15':>7}{'-20':>7}{'mean':>8}{'shared %':>13}")
    for name, (setting, r) in rows.items():
        m, x = r[stat], r[stat + "_x"]
        shared = r.get("shared")
        share = "" if shared is None or name == "cvx, none" else f"{100*shared[0]:.1f} / {100*shared[1]:.1f}"
        print(f"{name:<22}{setting:>30}{m['floor']:>8.2f}{'yes' if m['settled'] else 'NO':>6}"
              f"{m['reached']:>7d}{m['to_17']:>7d}{m['levels'][0]:>7d}{m['levels'][2]:>7d}{m['levels'][3]:>7d}"
              f"{m['rec']:>7d}{m['back_20']:>7d}{m['mean_db']:>8.2f}  {x['floor']:>7.2f}{x['levels'][2]:>7d}"
              f"{x['levels'][3]:>7d}{x['levels'][4]:>7d}{x['mean_db']:>8.2f}{share:>13}")
    print("   floors and means in dB, everything else in steps; own+3 (to the own floor + 3 dB) and -17 from zero at "
          "rest;\n   0, -10, -15, rec. (back to the own floor + 3 dB) and -20 dB after the flip; -1 = never within "
          "the run;\n   shared %: steps on which the scheme fired, rest run / change run")


def scenario_combination(ar_a, beta):
    """The two sKFs alone, combined by the review's convex combination with each scheme at every setting, and as our
    bank at every q; beside them the bank of six sKFs. The control and three tables."""
    start = time.time()
    sig = scenario_signals(ar_a, beta)
    (X_r, D_r), (X_c, D_c) = sig
    b_eta = noise_of(ar_a, beta)[1]
    den = ((clean_output(X_r)**2).mean(axis=0), (clean_output(X_c, change_at=N)**2).mean(axis=0))
    tag = f"ar{ar_a:g}_beta{beta:g}"
    eps_slow = tuned_values(ar_a, beta, b_eta, X_r, D_r)["sKF"]   # ours at -20 dB: given, or from the saved search
    pair = (EPS_FAST, eps_slow)
    print(f"b_eta = E|eta| = {b_eta:.4e}; the slow sKF eps = {eps_slow:.4e}, the fast one {EPS_FAST:.0e}")

    ours = {("sKF, exact", None): filter_run("sKF, exact", eps_slow, None, ar_a, beta, sig, b_eta, den)}
    for q in (Q_BANK, BEST_Q6[beta]):
        ours[("sKF, bank", q)] = filter_run("sKF, bank", np.nan, q, ar_a, beta, sig, b_eta, den)
    print("control: ours read from results/kf-bank-q-sweep/, against kf-bank-q-sweep.ipynb (ours / printed)")
    for (name, q), printed in PRINTED[beta].items():
        print(f"   {name:<11}{'' if q is None else f'q = {q:.3g}':<11}mean over the record "
              f"{ours[(name, q)]['mean']['mean_db']:.2f} / {printed:.2f}")

    def fast_run(X, D, f):
        mis, s = sKF_batch(X, D, ho, b_eta, np.full(len(X), EPS_FAST), chi_laplacian, flip_at=f, excess=True)
        return {"mis": mis, "s": s}

    def bank_run(q):
        def run(X, D, f):
            out = bank_batch(X, D, ho, [eps_slow, EPS_FAST], q, "sKF", chi_laplacian, log_density_laplacian, b_eta,
                             flip_at=f, excess=True)
            return {"mis": out["mis"], "s": out["s"], "mu": out["mu"]}   # mu[0]: the slow sKF, mu[1]: the fast one
        return run

    def cvx_run(sharing, mu_a, par, lam0):
        def run(X, D, f):
            out = cvx_batch(X, D, ho, b_eta, pair, mu_a, sharing, par, LAM0 if lam0 is None else lam0, flip_at=f,
                            excess=True)
            return {"mis": out["mis"], "s": out["s"], "mu": np.stack([1 - out["lam"], out["lam"]]),
                    "shared": out["shared"]}
        return run

    fast = new_run(f"fast-sKF_{EPS_FAST:.0e}_{tag}", fast_run, sig, den)
    sweep = {"bank, K = 2": {q: new_run(f"bank2_q{q:g}_{tag}", bank_run(q), sig, den) for q in Q_PAIR}}
    print(f"the fast sKF and the bank of the two done [{time.time() - start:.0f} s]", flush=True)
    for sharing, pars in SHARING.items():
        # the key of the first run's settings (lambda0 = 0.98) is kept, so that those runs are read, not run again
        sweep[sharing] = {(mu_a, par, lam0): new_run(f"cvx-{sharing}_mu{mu_a:g}_par{'-' if par is None else f'{par:g}'}"
                                                     + ("" if lam0 in (None, LAM0) else f"_lam0{lam0:g}") + f"_{tag}",
                                                     cvx_run(sharing, mu_a, par, lam0), sig, den)
                          for mu_a in MU_A_GRID for par, lam0 in pars}
        print(f"the combination, {sharing}, done [{time.time() - start:.0f} s]", flush=True)
    best = {name: min(runs, key=lambda k: runs[k]["mean"]["mean_db"]) for name, runs in sweep.items()}

    rows = {"slow sKF": (f"eps {eps_slow:.2e}", ours[("sKF, exact", None)]),
            "fast sKF": (f"eps {EPS_FAST:.0e}", fast)}
    for sharing in SHARING:
        rows[f"cvx, {sharing}"] = (setting_of(sharing, *best[sharing]), sweep[sharing][best[sharing]])
    rows["bank of the two, q = 0"] = ("q 0", sweep["bank, K = 2"][0.0])
    rows["bank of the two"] = (f"q {best['bank, K = 2']:.3g}", sweep["bank, K = 2"][best["bank, K = 2"]])
    rows["bank of six"] = (f"q {Q_BANK:.3g}", ours[("sKF, bank", Q_BANK)])
    rows["bank of six, best q"] = (f"q {BEST_Q6[beta]:.3g}", ours[("sKF, bank", BEST_Q6[beta])])
    print()
    show_settings(sweep, best)
    print()
    show_rows(rows, "mean")
    print()
    show_rows(rows, "median")
    new = [r["seconds"] for runs in sweep.values() for r in runs.values()] + [fast["seconds"]]
    print(f"[{time.time() - start:.0f} s; the {len(new)} new runs took {sum(new)/60:.0f} min when they were run]")
    return dict(rows=rows, sweep=sweep, best=best, eps_slow=eps_slow)


results = {}
print("comparison ready")

## 6. AR($-0.9$), $\beta^* = 0.2$

The scenario of the earlier notebooks.

In [ ]:
results[BETA] = scenario_combination(-0.9, BETA)

## 7. AR($-0.9$), Gaussian noise ($\beta^* = 2$)

The squared error is the right cost for this noise, and the Laplacian density that our bank's weights use is not. The
slow sKF is the deepest settled point of the search of `kf-bank-q-sweep.ipynb` ($-18.7$ dB), read from its saved search.

In [ ]:
results[BETA_GAUSS] = scenario_combination(-0.9, BETA_GAUSS)

## 8. Figures

**Figure 1.** Across the change, from 0.25 s before it to 2.5 s after it: misalignment (left) and excess error (right),
median over the 20 realisations. Rows: the two noises. Grey: the two sKFs alone (slow dashed, fast dotted). Colours:
the review's convex combination with each scheme at its best setting, our bank of the two sKFs at its best $q$, and our
bank of six sKFs at its best $q$. Grey line: $-20$ dB; black: the change.

**Figure 2.** The weight of the fast sKF, mean over the 20 realisations: $\lambda_t$ for the combination, $\mu_t$ of the
fast sKF for the bank of the two. Left: the whole change run (at rest, then the flip at 0); right: the window of
Figure 1. Every 8th step.

The colours are slots 1-6 of the categorical palette of the earlier notebooks, checked with the dataviz skill's
validator on white: all pass the adjacent colour-blind and normal-vision checks; aqua, yellow and magenta are below 3:1
contrast on white, so every plotted number is also in the tables of sections 6 and 7.

In [ ]:
# === copied from awu-gisv-vs-banks.ipynb section 10 (commit fab8cf0), which copied them from kf-bank-q-sweep.ipynb:
#     the greys and the axis helpers, NOT edited ===
SINGLE_GREY, TARGET_GREY, GRID_GREY, INK = "#52514e", "#a3a29b", "#e1e0d9", "#0b0b0b"


def window_time(curve):
    # time from the change [s] of the kept window of the change run
    return (np.arange(curve.shape[-1]) + WINDOW.start - N)/FS


def style_axes(ax, t_max, ylim):
    ax.axvline(0, color=INK, lw=0.8)                          # the change
    ax.axhline(TARGET_DB, color=TARGET_GREY, lw=0.8)          # -20 dB
    ax.set(xlim=(-0.25, t_max), ylim=ylim)
    ax.grid(color=GRID_GREY, lw=0.6)
    ax.set_axisbelow(True)


def db_range(curves, top=8.0):
    # y range of a row: from below the lowest median (5 dB steps) to top
    low = min(float(np.nanmin(10*np.log10(np.maximum(c, 1e-30)))) for c in curves)
    return (5*np.floor(low/5) - 2, top)
# === end of the copied block ===


T_MAX = 2.5
NOISE_LABEL = {BETA: "AR(-0.9), beta* = 0.2", BETA_GAUSS: "AR(-0.9), Gaussian noise (beta* = 2)"}
# slots 1-6 of the categorical palette, in order; the bank of the two takes slot 2, the sKF family's orange
LOOK = {"slow sKF": (SINGLE_GREY, "--"), "fast sKF": (SINGLE_GREY, ":"),
        "cvx, none": ("#2a78d6", "-"), "bank of the two": ("#eb6834", "-"), "cvx, gradual": ("#1baf7a", "-"),
        "cvx, copy": ("#eda100", "-"), "cvx, feedback": ("#e87ba4", "-"), "bank of six, best q": ("#008300", "-")}
WEIGHTS = ["cvx, none", "bank of the two", "cvx, gradual", "cvx, copy", "cvx, feedback"]


def label_of(name, setting):
    return f"{name} ({setting})"


def figure_curves(results):
    """Figure 1: misalignment and excess error across the change, median."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10), constrained_layout=True, sharey="row")
    for row, beta in enumerate((BETA, BETA_GAUSS)):
        rows = results[beta]["rows"]
        t = window_time(rows["slow sKF"][1]["curves"]["mis_q"][1])
        keep = t <= T_MAX
        ylim = db_range([rows[name][1]["curves"][f"{metric}_q"][1][keep] for name in LOOK for metric in ("mis", "exc")])
        for col, metric in enumerate(("mis", "exc")):
            ax = axes[row, col]
            for name, (colour, style) in LOOK.items():
                setting, r = rows[name]
                ax.plot(t[keep], 10*np.log10(r["curves"][f"{metric}_q"][1][keep]), color=colour, ls=style, lw=1.5,
                        label=label_of(name, setting))
            style_axes(ax, T_MAX, ylim)
            ax.set_title(f"{NOISE_LABEL[beta]}: " + ("misalignment" if metric == "mis" else "excess error"),
                         fontsize=10)
            ax.set_xlabel("time from the change [s]")
            ax.legend(fontsize=8, loc="upper right")
        axes[row, 0].set_ylabel("median [dB]")
    fig.suptitle("The two sKFs, the review's convex combination (each scheme at its best setting) and our banks: median")
    plt.show()


def figure_weights(results):
    """Figure 2: the weight of the fast sKF, mean over the realisations."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 9), constrained_layout=True, sharey=True)
    for row, beta in enumerate((BETA, BETA_GAUSS)):
        rows = results[beta]["rows"]
        for col, (t_lo, t_hi) in enumerate(((-N/FS, N/FS), (-0.25, T_MAX))):
            ax = axes[row, col]
            for name in WEIGHTS:
                setting, r = rows[name]
                mu = r["curves"]["mu"][1]                    # the fast sKF; every 8th step of the change run
                t = (np.arange(mu.size)*8 - N)/FS
                keep = (t >= t_lo) & (t <= t_hi)
                ax.plot(t[keep], mu[keep], color=LOOK[name][0], lw=1.2, label=label_of(name, setting))
            ax.axvline(0, color=INK, lw=0.8)
            ax.set(xlim=(t_lo, t_hi), ylim=(-0.02, 1.02))
            ax.grid(color=GRID_GREY, lw=0.6)
            ax.set_axisbelow(True)
            ax.set_title(f"{NOISE_LABEL[beta]}: " + ("the whole change run" if col == 0 else "around the change"),
                         fontsize=10)
            ax.set_xlabel("time from the change [s]")
        axes[row, 0].set_ylabel("weight of the fast sKF, mean")
        axes[row, 1].legend(fontsize=8, loc="center right")
    fig.suptitle("The weight of the fast sKF: lambda_t of the combination, mu_t of the fast sKF in the bank of the two")
    plt.show()


figure_curves(results)

In [ ]:
figure_weights(results)

## 9. Findings

**Checks: all pass.**
1. Inside the combination each sKF is `sKF_batch` bit for bit, and gradual transfer with $\ell = 1$, simple copy and
   feedback with $N_0$ longer than the run give "none" bit for bit (4.1).
2. `cvx_batch` against Tables 4 and 5 line by line: the output's misalignment within $1.5 \times 10^{-9}$, the same size
   as the one-ulp yardstick ($1.9 \times 10^{-9}$). $\lambda_t$ differs by up to 0.96 on a handful of steps, as it does
   between the reference and itself one ulp apart: in the first steps, while the two filters are nearly equal, $a_t$
   jumps between $\pm a^+$ (4.2).
3. The bank of the two at $q = 0$: each sKF is `sKF_batch` bit for bit (4.3).
4. Control: the slow sKF and the bank of six, read from `results/kf-bank-q-sweep/`, give the means printed by
   `kf-bank-q-sweep.ipynb` exactly, in both noises.

**1. Heavy-tailed noise ($\beta^* = 0.2$): with the same two sKFs, our bank is ahead of the review's combination, but
by little.** Mean curves:

| filter | best setting | floor | $-15$ dB after the flip | back to $-20$ dB | mean over the record |
|---|---|---|---|---|---|
| slow sKF alone | $\varepsilon = 1.35 \times 10^{-6}$ | $-20.01$ | 6 497 | 12 224 | $-14.08$ |
| fast sKF alone | $\varepsilon = 10^{-4}$ | $-6.21$ | never | never | $-5.96$ |
| combination, none | $\mu_a = 0.3$ | $-19.68$ | 6 490 | 13 129 | $-14.68$ |
| combination, gradual transfer | $\mu_a = 1$, $\ell = 0.9$, $\lambda_0 = 0.7$ | $-19.64$ | 4 909 | 13 126 | $-15.00$ |
| combination, simple copy | $\mu_a = 1$, $N_0 = 10$, $\lambda_0 = 0.7$ | $-19.64$ | 4 896 | 13 126 | $-15.00$ |
| combination, feedback | $\mu_a = 1$, $N_0 = 100$ | $-19.05$ | 5 516 | never | $-14.67$ |
| **our bank of the two** | $q = 10^{-4}$ | $-19.85$ | **4 605** | **11 140** | $\mathbf{-15.28}$ |
| our bank of six | $q = 0.1$ | $-27.47$ | 5 300 | 8 464 | $-16.52$ |

- Our bank of the two is 0.28 dB ahead of the best combination on the mean over the record (0.23 dB on the median
  curves), reaches $-15$ dB after the flip 6 % sooner and $-20$ dB 15 % sooner. The floors of the two-filter methods are
  within 0.4 dB of each other, except feedback's (0.6 to 0.8 dB higher).
- By the excess error the two-filter methods are nearly level: means $-16.8$ to $-17.1$ dB ($-16.4$ for feedback), the
  slow sKF alone $-17.01$.
- Here the best settings of gradual transfer and simple copy are inside the grid ($\mu_a = 3$ and $\lambda_0 = 0.5$ are
  no better). Feedback's best is at $N_0 = 100$, the largest on its grid, in both noises; larger $N_0$ moves it towards
  "none", which it already equals here.
- The bank of six at its best $q$ is still the best filter here, but it has other filters ($\varepsilon$ from $10^{-9}$
  to $10^{-4}$), so it is not a like-for-like comparison.

**2. Gaussian noise ($\beta^* = 2$): the review's combination is ahead of our banks.** Mean curves:

| filter | best setting | floor | $-10$ dB after the flip | $-15$ dB after the flip | mean over the record |
|---|---|---|---|---|---|
| slow sKF alone | $\varepsilon = 10^{-7}$ | $-18.69$ | 41 453 | 62 351 | $-6.77$ |
| fast sKF alone | $\varepsilon = 10^{-4}$ | $-6.34$ | never | never | $-6.02$ |
| combination, none | $\mu_a = 0.3$ | $-18.58$ | 40 927 | 62 004 | $-9.14$ |
| combination, gradual transfer | $\mu_a = 3$, $\ell = 0.9$, $\lambda_0 = 0.7$ | $-16.40$ | 10 059 | 28 259 | $-12.22$ |
| **combination, simple copy** | $\mu_a = 1$, $N_0 = 10$, $\lambda_0 = 0.5$ | $-18.56$ | 12 663 | 32 082 | $\mathbf{-12.23}$ |
| combination, feedback | $\mu_a = 3$, $N_0 = 100$ | $-17.55$ | 21 222 | 36 786 | $-10.78$ |
| our bank of the two | $q = 0.01$ | $-12.23$ | **8 492** | never | $-10.49$ |
| our bank of six | $q = 0.01$ | $-17.88$ | 23 880 | 42 347 | $-10.53$ |

- Gradual transfer and simple copy are 1.7 dB ahead of our bank of the two and of our bank of six at its best $q$ on
  the mean over the record (1.8 to 2.0 dB on the median curves). By the mean excess error they are 4.5 dB ahead of our
  bank of the two ($-15.6$ against $-11.1$ dB) and 0.7 dB ahead of the bank of six at its best $q$; the bank of six at
  $q = 10^{-4}$ has the best mean excess error of all ($-16.52$ dB), with a worse misalignment.
- Our bank of the two is the fastest to $-10$ dB after the flip, but its floor at rest is $-12.2$ dB, 6.5 dB above the
  slow sKF's: it keeps 13 % of its weight on the fast sKF at rest (Figure 2; mean over the second half of the rest
  part), where the combination keeps 2 to 3 %, near the 1.8 % floor that $\mathrm{sgm}(-4)$ allows. With $q = 10^{-4}$ the
  bank keeps 0.3 %, but then reaches $-10$ dB after the flip only after 31 233 steps (mean $-9.61$ dB). Probably the
  Laplacian density in the weights under Gaussian noise; not checked here.
- Here the best settings sit at an edge of the grid ($\mu_a = 3$ for gradual transfer, $\lambda_0 = 0.5$ for simple copy).
  The last widening of the grid gained 0.27 dB, and five other settings are within 0.15 dB of the best, so the top is
  flat.

**3. Sharing: our bank gains only through it, and the review's threshold barely lets the combination's act.**
- Without sharing our bank is the slow sKF: at $q = 0$ it gives the slow sKF's numbers in every column, in both noises.
  The fast sKF's weight is 0.000 at rest and stays 0.000 after the flip: the evidence piled up at rest is not undone. Without sharing the
  combination gains 0.6 dB ($\beta^* = 0.2$) and 2.4 dB ($\beta^* = 2$) over the slow sKF. Our bank's gain is all in the
  mixing step.
- With the review's threshold ($\lambda_0 = 0.98$, near $\mathrm{sgm}(a^+) = 0.982$) gradual transfer and simple copy fired
  on about as many steps in the rest run as in the change run, so almost never after the flip, and gave the numbers of
  "none" for $\mu_a \le 1$ (Gaussian noise) or $\le 0.3$ ($\beta^* = 0.2$). After the flip the mean of $\lambda_t$ over the
  realisations peaks near 0.9 (Figure 2). They help once $\lambda_0$ is 0.9 or lower and $\mu_a \ge 1$: then they gain up
  to 0.3 dB ($\beta^* = 0.2$) and 3.1 dB ($\beta^* = 2$) over "none".
- Gradual transfer and simple copy are one scheme in practice: their best means differ by 0.01 dB at most.
  Feedback is the weakest of the three.
- So the combination needs three numbers tuned here ($\mu_a$, $\lambda_0$, and $\ell$ or $N_0$), besides $a^+$ and $\eta$,
  and its best $\lambda_0$ is far from the review's example (0.982). Our bank of the two needs one ($q$), but its best $q$ also
  changes with the noise ($10^{-4}$ and $10^{-2}$).

**Not done.**
- Our bank with the Gaussian density in its weights under Gaussian noise, the like-for-like test for item 2: the
  combination's cost is the right one there and our weights' density is not.
- More than two filters: the review's softmax combination of the six sKFs against our bank of six.
- The affine combination, KF components, white input, a robust mixing rule for impulsive noise, and the review's own
  figures.

In [ ]:
new_runs = sum(r["seconds"] for res in results.values() for runs in res["sweep"].values() for r in runs.values())
print(f"whole notebook: {(time.time() - NOTEBOOK_START)/60:.1f} min; the new runs (without the fast sKF) took "
      f"{new_runs/60:.1f} min when they were run")